In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow

# Update this path to your sleep-scored parquet file
parquet_path = Path("/mnt/harris/hypnose/hypnose_eeg/derivatives/sub-066_id-427/ses-001_date-20260717/saved_results/sub-066_ses-001_recording-001_2026-07-17_13_40_01_somnotate_predictions.parquet")

sleep_df = pd.read_parquet(parquet_path)

print(f"Loaded {parquet_path}")
print(f"Rows: {len(sleep_df):,} | Columns: {sleep_df.shape[1]}")
sleep_df.head()

In [ ]:
import mne
fif_path = Path("/mnt/harris/hypnose/hypnose_eeg/derivatives/sub-066_id-427/ses-001_date-20260717/ephys/sub-066_ses-001_recording-001_2026-07-17_13_40_01_resampled-128hz_raw.fif")
raw = mne.io.read_raw_fif(fif_path, preload=False, verbose="ERROR")

print(f"Loaded .fif: {fif_path}")
print(raw)

In [ ]:
raw

In [ ]:
import numpy as np

# Compute non-overlapping 4-second delta power while retaining EEG/EMG channels.
if "aligned_raw" not in globals():
    eeg_emg_picks = mne.pick_types(raw.info, eeg=True, emg=True, exclude=[])
    if len(eeg_emg_picks) == 0:
        raise ValueError("The FIF file contains no channels typed as EEG or EMG.")
    aligned_raw = raw.copy().pick(eeg_emg_picks)
epoch_seconds = 4.0
sfreq = float(aligned_raw.info["sfreq"])
epoch_samples = int(round(epoch_seconds * sfreq))
n_epochs = aligned_raw.n_times // epoch_samples

scores = sleep_df[["time_s", "label_model"]].copy()
scores["epoch_id"] = np.floor(scores["time_s"] / epoch_seconds).astype("int64")
score_counts = scores.groupby("epoch_id").size()
complete_score_epochs = score_counts[score_counts == int(epoch_seconds)].index
epoch_labels = (
    scores[scores["epoch_id"].isin(complete_score_epochs)]
    .groupby("epoch_id")["label_model"]
    .agg(lambda values: values.mode().iloc[0])
)

# Welch settings give sufficient resolution around the 0.5--4 Hz delta band.
n_per_seg = min(epoch_samples, int(round(2 * sfreq)))
n_fft = 1 << (n_per_seg - 1).bit_length()
n_overlap = n_per_seg // 2
rows = []
chunk_epochs = 512
for first_epoch in range(0, n_epochs, chunk_epochs):
    last_epoch = min(first_epoch + chunk_epochs, n_epochs)
    start = first_epoch * epoch_samples
    stop = last_epoch * epoch_samples
    samples = aligned_raw.get_data(start=start, stop=stop)
    batch = samples.reshape(len(aligned_raw.ch_names), last_epoch - first_epoch, epoch_samples)
    batch = np.moveaxis(batch, 1, 0)  # epochs x channels x samples
    batch *= 1e6  # MNE stores EEG/EMG in volts; calculate PSD in microvolts.
    psd, frequencies = mne.time_frequency.psd_array_welch(
        batch, sfreq=sfreq, fmin=0.5, fmax=4.0,
        n_fft=n_fft, n_per_seg=n_per_seg, n_overlap=n_overlap, verbose=False,
    )
    delta_power = np.trapezoid(psd, frequencies, axis=-1)
    for offset, power in enumerate(delta_power):
        epoch_id = first_epoch + offset
        if epoch_id not in epoch_labels.index:
            continue
        rows.append({
            "epoch_id": epoch_id,
            "time_from_recording_start_s": epoch_id * epoch_seconds,
            "label_model": epoch_labels.loc[epoch_id],
            **dict(zip(aligned_raw.ch_names, power)),
        })

delta_power_df = pd.DataFrame(rows)
print(f"Computed {len(delta_power_df):,} complete 4-second epochs")
print(f"Channels: {aligned_raw.ch_names} (delta power in uV^2)")
delta_power_df.head()

In [ ]:
power_cols = [c for c in delta_power_df.columns if c not in ["epoch_id", "time_from_recording_start_s", "label_model"]]

mean_delta_by_label = delta_power_df.groupby("label_model")[power_cols].mean()
print(mean_delta_by_label)

In [ ]:
# Count occurrences where label_model == 0 in the per-second scoring table
count_label0 = (sleep_df["label_model"] == 0).sum()
print(f"label_model == 0 appears {count_label0:,} times in sleep_df")

# Optional: same count in 4-second epoch table
count_label0_epochs = (delta_power_df["label_model"] == 0).sum()
print(f"label_model == 0 appears {count_label0_epochs:,} times in delta_power_df")